# 02 — Replication: Paper 2 (Safety Neurons)

**Goal:** Reproduce the core finding of:
  "Towards Understanding Safety Alignment: A Mechanistic Perspective
   from Safety Neurons"

By the end of this notebook we will have:
  1. Neuron activations cached for both models on the same prompts
  2. Change scores for every neuron across all layers
  3. A ranked list of top safety neurons
  4. Causal effect scores for increasing k (how many neurons explain alignment)
  5. Confirmation that safety neurons are special vs random neurons

These results are the baseline for Experiments 1–3.
Save `top_neurons` and `scores` — later notebooks depend on them.

**Key methodological note:**
  Both models must be run on IDENTICAL input text.
  We generate text with the base model first, then feed that same text
  to both models. This ensures activation differences come from the
  model weights, not from different outputs.

**Expected runtime:** ~40 minutes total
  - Activation collection (base)     : ~15 min
  - Activation collection (instruct) : ~15 min
  - Causal effect scoring            : ~10 min

## 0. Imports

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/from-neurons-to-directions/src")

import torch
import matplotlib.pyplot as plt

from model_utils import (
    load_model_and_tokenizer,
    apply_chat_template,
    get_num_layers,
    generate,
)
from activation_store import ActivationStore
from safety_neurons import (
    collect_generated_span_activations,
    compute_change_scores_rms,
    get_top_safety_neurons,
    compute_causal_effect,
    generate_with_neuron_ablation,
)
from metrics import refusal_rate
from viz import (
    plot_change_score_heatmap,
    plot_neuron_layer_dist,
    plot_causal_effect_bar,
    plot_refusal_rates,
)

print("Imports OK")

## 1. Load base models

Unlike notebook 01 (which only needed the instruct model), Paper 2's
method requires BOTH models simultaneously — we contrast their
activations on the same inputs to find which neurons changed. For memory limitation issues, we first load base model, interact with it, delete it, then repeat the process with instruct version.

In [ ]:
print("Loading base model...")
base_model, base_tokenizer = load_model_and_tokenizer("qwen_base")
n_layers = get_num_layers(base_model)

print(f"\nModel loaded | {n_layers} layers")

## 2. Load prompts from notebook 01

We reuse the same harmful and harmless prompts from notebook 01.
Consistency matters — the prompt sets should be identical across
all notebooks so results are comparable.

In production, load these from a saved file rather than redefining
them. Here we redefine the raw lists for self-containedness, then
apply the appropriate formatting per model.

In [ ]:
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF-READ")

harmful_ds = load_dataset("walledai/AdvBench", token=hf_token)
harmless_ds = load_dataset("tatsu-lab/alpaca", token=hf_token)

harmful = harmful_ds["train"]
harmless = harmless_ds["train"]

harmful_prompts_raw = harmful["prompt"]

# Keep only Alpaca samples with empty input field
harmless_prompts_raw = [
    instruction
    for instruction, inp in zip(harmless["instruction"], harmless["input"])
    if inp.strip() == ""
]

n = min(len(harmful_prompts_raw), len(harmless_prompts_raw))

harmful_prompts_raw = harmful_prompts_raw[:n]
harmless_prompts_raw = harmless_prompts_raw[:n]

print(f"Harmful prompts  : {len(harmful_prompts_raw)}")
print(f"Harmless prompts : {len(harmless_prompts_raw)}")

## 3. Generate shared text with the base model

Paper 2's critical methodological insight: we must feed BOTH models
the same input text. If we used the base model's outputs as inputs
to both, we control for output differences.

Concretely: generate responses with the base model, then use those
(prompt + response) pairs as inputs when collecting activations from
both models. This way any neuron activation difference is purely due
to the model weights, not the input distribution.

We use the raw prompts (no chat template) for the base model since
it was not trained with one.

Memory tip: if you only have 16 GB VRAM, do this instead:
  1. Collect base activations here, save them
  2. del base_model; torch.cuda.empty_cache()
  3. Load instruct model, collect instruct activations, save them
  4. Load both saved .pt files for the contrast step

In [ ]:
print("Generating shared text with base model...")

# Use raw prompts for base model (no chat template)
shared_inputs = harmful_prompts_raw + harmless_prompts_raw

# Generate base model completions — these become the shared text both
# models are evaluated on. Keep responses short (50 tokens) so the
# total sequence fits in memory easily.
base_completions = generate(
    base_model, base_tokenizer,
    shared_inputs,
    max_new_tokens=50,
)

# Concatenate prompt + completion to form the shared evaluation text
shared_texts = [
    prompt + " " + completion
    for prompt, completion in zip(shared_inputs, base_completions)
]

print(f"Generated {len(shared_texts)} shared texts.")
print(f"\nExample shared text:\n{shared_texts[0][:300]}...")

# Split back into harmful and harmless
shared_harmful  = shared_texts[:len(harmful_prompts_raw)]
shared_harmless = shared_texts[len(harmful_prompts_raw):]

## 4. Collect neuron activations — base model

We hook the pre-down-proj MLP activations (Paper 2's "neuron activations")
at every layer. For each prompt we store the last token's activation
vector — shape [intermediate_size] = [14336] for Llama 8B.

These are saved to disk; no need to re-run if the kernel restarts.

In [ ]:
# base_store = ActivationStore(
#     base_model, base_tokenizer,
#     save_dir="/kaggle/working/from-neurons-to-directions/activations/base"
# )
# all_layers = list(range(n_layers))

# print("Collecting base model neuron activations (harmful)...")
# base_store.collect(
#     prompts=shared_harmful,
#     mode="neurons",
#     layers=all_layers,
#     tag="harmful",
#     batch_size=64,
#     token_position=-1,
# )
# base_store.save()

# print("\nCollecting base model neuron activations (harmless)...")
# base_store.collect(
#     prompts=shared_harmless,
#     mode="neurons",
#     layers=all_layers,
#     tag="harmless",
#     batch_size=64,
#     token_position=-1,
# )
# base_store.save()

# print("\nBase activations saved.")

# Manually setting layer for best layer, change later if necessary

collect_generated_span_activations(
    base_model, base_tokenizer,
    prompts_raw=shared_harmful,   # format_for_base(p) outputs
    completions=base_completions,                # from section 3, same order
    layers=[16],
    save_path="/kaggle/working/harmful_genspan_base.pt",
)

In [ ]:
import gc

print(f"GPU memory before cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

del base_model
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after cleanup:  {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5. Collect neuron activations — instruct model

Same shared texts, same layers, same token position.
The only difference is which model processes them.

In [ ]:
print("\nLoading instruct model...")
instruct_model, instruct_tokenizer = load_model_and_tokenizer("qwen_instruct")

# instruct_store = ActivationStore(
#     instruct_model, instruct_tokenizer,
#     save_dir="/kaggle/working/from-neurons-to-directions/activations/instruct"
# )

# print("Collecting instruct model neuron activations (harmful)...")
# instruct_store.collect(
#     prompts=shared_harmful,
#     mode="neurons",
#     layers=all_layers,
#     tag="harmful",
#     batch_size=4,
#     token_position=-1,
# )
# instruct_store.save()

# print("\nCollecting instruct model neuron activations (harmless)...")
# instruct_store.collect(
#     prompts=shared_harmless,
#     mode="neurons",
#     layers=all_layers,
#     tag="harmless",
#     batch_size=4,
#     token_position=-1,
# )
# instruct_store.save()

# print("\nInstruct activations saved.")

collect_generated_span_activations(
    instruct_model, instruct_tokenizer,
    prompts_raw=shared_harmful,   # identical input → identical tokenization
    completions=base_completions,
    layers=[16],
    save_path="/kaggle/working/harmful_genspan_instruct.pt",
)

## 6. Compute change scores

For each neuron at each layer, compute how much its mean activation
changed between the base and instruct model on the same inputs.

score(neuron j, layer l) = |mean_instruct(j,l) - mean_base(j,l)|

High score → this neuron's behavior changed substantially after alignment
Low score  → alignment didn't affect this neuron much

We use the harmful prompts for contrasting — safety neurons should
show the biggest difference precisely on inputs that trigger safety behavior.

In [ ]:
# base_harmful_acts     = ActivationStore.load(
#     "/kaggle/working/from-neurons-to-directions/activations/base/harmful_neurons.pt"
# )
# instruct_harmful_acts = ActivationStore.load(
#     "/kaggle/working/from-neurons-to-directions/activations/instruct/harmful_neurons.pt"
# )

# scores = compute_change_scores(
#     base_acts=base_harmful_acts,
#     instruct_acts=instruct_harmful_acts,
#     method="mean_diff",
# )

scores = compute_change_scores_rms(
    base_activations_path="/kaggle/working/harmful_genspan_base.pt",
    instruct_activations_path="/kaggle/working/harmful_genspan_instruct.pt",
    layers=[16],
)

print(f"\nChange scores computed for {len(scores)} layers.")

# Quick stats: which layer has the highest average change?
avg_scores = {l: scores[l].mean().item() for l in scores}
top_layer  = max(avg_scores, key=avg_scores.get)
print(f"Highest average change: layer {top_layer} "
      f"(mean score = {avg_scores[top_layer]:.4f})")

# Save scores for later notebooks
torch.save(scores, "/kaggle/working/from-neurons-to-directions/change_scores.pt")
print("Saved → /kaggle/working/from-neurons-to-directions/change_scores.pt")

## 7. Visualize change scores

Heatmap: rows = layers, columns = top neurons per layer by change score.
Color intensity = how much that neuron changed after alignment.

What to look for:
  - Do changes cluster in specific layers? (localization)
  - Are a few neurons far brighter than the rest? (sparsity)

In [ ]:
fig = plot_change_score_heatmap(
    scores=scores,
    top_k_neurons=100,
    title="Neuron Change Scores: Base → Instruct (Harmful Prompts)",
    save_path="/kaggle/working/p2_change_heatmap.png",
)
plt.show()

## 8. Select top safety neurons

Rank all neurons globally by change score and take the top k.
We start with k=500 (~0.1% of all neurons) for efficiency,
then scale up in the causal effect analysis.

In [ ]:
top_neurons = get_top_safety_neurons(scores, k=500)

print(f"\nTop-500 safety neurons selected.")
print(f"Example (layer, neuron_idx): {top_neurons[:5]}")

# Save for later notebooks
torch.save(top_neurons, "/kaggle/working/from-neurons-to-directions/top_neurons.pt")
print("Saved → /kaggle/working/from-neurons-to-directions/top_neurons.pt")

## 9. Visualize neuron layer distribution

How are the top safety neurons distributed across layers?
Paper 2 finds they cluster in mid-to-late layers.
If you see a similar pattern, that's a replication signal.

In [ ]:
fig = plot_neuron_layer_dist(
    safety_neurons=top_neurons,
    n_layers=n_layers,
    title="Distribution of Top-500 Safety Neurons Across Layers",
    save_path="/kaggle/working/p2_neuron_dist.png",
)
plt.show()

## 10. Causal effect analysis

The central question: do these neurons actually CAUSE safety behavior?

We measure this by injecting the instruct model's safety neuron
activations into the base model at inference time (dynamic activation
patching) and checking how much of the instruct model's refusal
behavior is recovered.

C = patched_refusal_rate / instruct_refusal_rate

C ≈ 1.0 → these neurons fully explain alignment
C ≈ 0.0 → no causal role

We compute C for increasing k to find the "elbow" — the point where
adding more neurons stops helping. Paper 2 finds the elbow around 5%.

This is slow (~10 min). Use n_eval=10 for a quick check.

In [ ]:
# Try fitting base model back in
base_model, base_tokenizer = load_model_and_tokenizer("qwen_base")

In [ ]:
k_values = [50, 100, 250, 500]
causal_effects = {}

for k in k_values:
    print(f"\n--- Evaluating k={k} neurons ---")
    top_k = get_top_safety_neurons(scores, k=k)
    C = compute_causal_effect(
        base_model=base_model,
        instruct_model=instruct_model,
        tokenizer=base_tokenizer,
        harmful_prompts=harmful_prompts_raw,
        safety_neurons=top_k,
        n_eval=10,       # increase to 20 for final results
    )
    causal_effects[k] = C

torch.save(causal_effects, "/kaggle/working/from-neurons-to-directions/causal_effects.pt")

fig = plot_causal_effect_bar(
    causal_effects=causal_effects,
    title="Causal Effect of Top-k Safety Neurons (Paper 2 Replication)",
    save_path="/kaggle/working/p2_causal_effect.png",
)
plt.show()

## 11. Safety neurons vs random neurons

Paper 2 claims safety neurons are special — not just any neurons
would produce the same causal effect.

We verify this by picking a random set of neurons (same size as our
top-500) and measuring their causal effect. If the random baseline
is near zero and the safety neurons are high, the neurons are special.

In [ ]:
import random
random.seed(42)

from model_utils import get_intermediate_size
intermediate_size = get_intermediate_size(base_model)

# Random neurons: same count as top_neurons, randomly sampled
random_neurons = [
    (random.randint(0, n_layers - 1), random.randint(0, intermediate_size - 1))
    for _ in range(500)
]

print("Computing causal effect for RANDOM neurons (baseline)...")
C_random = compute_causal_effect(
    base_model=base_model,
    instruct_model=instruct_model,
    tokenizer=base_tokenizer,
    harmful_prompts=harmful_prompts_raw,
    safety_neurons=random_neurons,
    n_eval=10,
)

C_safety = causal_effects.get(500, None)
if C_safety is None:
    print("Run section 10 first to get C_safety.")
else:
    print(f"\nCausal Effect — Safety neurons (k=500) : {C_safety:.3f}")
    print(f"Causal Effect — Random  neurons (k=500) : {C_random:.3f}")
    print(f"Safety neurons are {C_safety / max(C_random, 1e-6):.1f}x more effective than random.")

## 12. Verify: neuron ablation suppresses refusal

The mirror of causal effect: if safety neurons cause safety,
then ablating them (zeroing their activations) in the instruct model
should suppress refusal — just like removing the refusal direction did
in notebook 01.

This result is also used in Experiment 2 (notebook 04):
we collect residual stream activations DURING this ablation to see
whether the refusal direction disappears.

In [ ]:
print("Testing safety neuron ablation on instruct model...")

# Baseline: instruct model with no intervention
baseline_responses = generate(
    instruct_model, instruct_tokenizer,
    [apply_chat_template(p, instruct_tokenizer) for p in harmful_prompts_raw[:20]],
    max_new_tokens=100,
)
baseline_rate = refusal_rate(baseline_responses)

# Ablated: instruct model with safety neurons zeroed out
ablated_responses = generate_with_neuron_ablation(
    model=instruct_model,
    tokenizer=instruct_tokenizer,
    prompts=[apply_chat_template(p, instruct_tokenizer) for p in harmful_prompts_raw[:20]],
    safety_neurons=top_neurons,
    max_new_tokens=100,
)
ablated_rate = refusal_rate(ablated_responses)

print(f"\nRefusal rate — baseline (instruct)   : {baseline_rate:.0%}")
print(f"Refusal rate — after neuron ablation : {ablated_rate:.0%}")
print(f"Delta                                : {ablated_rate - baseline_rate:+.0%}")

# Plot comparison
fig = plot_refusal_rates(
    conditions={
        "Instruct model (no intervention)": baseline_rate,
        "Instruct + safety neurons ablated": ablated_rate,
    },
    title="Effect of Safety Neuron Ablation on Refusal Rate",
    save_path="/kaggle/working/p2_neuron_ablation.png",
)
plt.show()

## 13. Summary

In [ ]:
print("=" * 55)
print("PAPER 2 REPLICATION SUMMARY")
print("=" * 55)
print(f"Model pair         : Llama-3.1-8B (base) vs Instruct")
print(f"Shared prompts     : {len(shared_harmful)} harmful, {len(shared_harmless)} harmless")
print(f"Top safety neurons : {len(top_neurons)}")
print(f"Causal effects     : { {k: f'{v:.2f}' for k, v in causal_effects.items()} }")
print(f"Random baseline    : {C_random:.3f}")
print(f"Baseline refusal   : {baseline_rate:.0%}")
print(f"After ablation     : {ablated_rate:.0%}")
print("=" * 55)
print("\nFiles saved:")
print("  results/change_scores.pt")
print("  results/top_neurons.pt")
print("  results/causal_effects.pt")
print("  results/activations/base/harmful_neurons.pt")
print("  results/activations/base/harmless_neurons.pt")
print("  results/activations/instruct/harmful_neurons.pt")
print("  results/activations/instruct/harmless_neurons.pt")
print("  results/figures/p2_change_heatmap.png")
print("  results/figures/p2_neuron_dist.png")
print("  results/figures/p2_causal_effect.png")
print("  results/figures/p2_neuron_ablation.png")
print("\nNext: run 03_exp1_geometric.ipynb")